### **1. Installing and importing**

In [12]:
!pip install requests pandas python-dotenv tqdm

In [13]:
import requests
import pandas as pd
import time
import os
from dotenv import load_dotenv
from tqdm import tqdm

### **2. Loading key**

In [14]:
load_dotenv()
API_KEY = os.getenv("TMDB_API_KEY")
BASE_URL = "https://api.themoviedb.org/3"

### **3. Testing a single cell before looping**

In [15]:
resp = requests.get(
    f"{BASE_URL}/movie/550",
    params={"api_key": API_KEY, "language": "en-US"}
)
print(resp.status_code)
resp.json()

200


{'adult': False,
 'backdrop_path': '/c6OLXfKAk5BKeR6broC8pYiCquX.jpg',
 'belongs_to_collection': None,
 'budget': 63000000,
 'genres': [{'id': 18, 'name': 'Drama'}, {'id': 53, 'name': 'Thriller'}],
 'homepage': 'https://www.20thcenturystudios.com/movies/fight-club',
 'id': 550,
 'imdb_id': 'tt0137523',
 'origin_country': ['US'],
 'original_language': 'en',
 'original_title': 'Fight Club',
 'overview': 'A ticking-time-bomb insomniac and a slippery soap salesman channel primal male aggression into a shocking new form of therapy. Their concept catches on, with underground "fight clubs" forming in every town, until an eccentric gets in the way and ignites an out-of-control spiral toward oblivion.',
 'popularity': 46.8587,
 'poster_path': '/jSziioSwPVrOy9Yow3XhWIBDjq1.jpg',
 'production_companies': [{'id': 711,
   'logo_path': '/tEiIH5QesdheJmDAqQwvtN60727.png',
   'name': 'Fox 2000 Pictures',
   'origin_country': 'US'},
  {'id': 508,
   'logo_path': '/4sGWXoboEkWPphI6es6rTmqkCBh.png',
   '

### **4. Wraping the call in a function**

In [16]:
def get_movie(movie_id):
    resp = requests.get(
        f"{BASE_URL}/movie/{movie_id}",
        params={"api_key": API_KEY, "language": "en-US"}
    )
    resp.raise_for_status()
    return resp.json()

### **5. Looping over many IDs, with pacing and error handling**

In [17]:
movie_ids = range(1, 3000)
results = []

for id in tqdm(movie_ids):
    resp = requests.get(f"{BASE_URL}/movie/{id}", params={"api_key": API_KEY})
    if resp.status_code == 200:
        results.append(resp.json())
    time.sleep(0.05)

  0%|          | 0/2999 [00:00<?, ?it/s]

100%|██████████| 2999/2999 [06:53<00:00,  7.26it/s]


### **6. Flattening the JSON into a DataFrame**

In [22]:
rows = []
for m in results:
    rows.append({
        "id": m["id"],
        "title": m["title"],
        "release_date": m.get("release_date"),
        "genres": [g["name"] for g in m.get("genres", [])],
        "budget": m.get("budget"),
        "revenue": m.get("revenue"),
        "vote_average": m.get("vote_average"),
        "vote_count": m.get("vote_count"),
    })

df = pd.DataFrame(rows)
df

,id,title,release_date,genres,budget,revenue,vote_average,vote_count
0,2,Ariel,1988-10-21,"[Drama, Romance, Crime]",0,0,7.115,384
1,3,Shadows in Paradise,1986-10-17,"[Drama, Romance]",0,0,7.264,449
2,5,Four Rooms,1995-12-09,[Comedy],4000000,4257354,5.919,2909
3,6,Judgment Night,1993-10-15,"[Action, Crime, Drama, Thriller]",21000000,12136938,6.446,389
4,8,Life in Loops (A Megacities RMX),2006-01-01,[Documentary],42000,0,7.300,34
...,...,...,...,...,...,...,...,...
1953,2994,Danger: Diabolik,1968-01-24,"[Action, Comedy, Crime, Thriller]",400000,0,6.300,262
1954,2995,Masquerade,1965-04-13,"[Comedy, Adventure, Thriller]",0,0,5.028,18
1955,2996,The Lost World,1998-08-01,"[Science Fiction, TV Movie]",0,0,6.000,1
1956,2997,The Lost World,1998-10-13,"[TV Movie, Adventure, Science Fiction]",0,0,4.700,22


In [25]:
def get_imdb_id(movie_id):
    resp = requests.get(
        f"{BASE_URL}/movie/{movie_id}/external_ids",
        params={"api_key": API_KEY}
    )
    if resp.status_code == 200:
        return resp.json().get("imdb_id")
    return None

df["imdb_id"] = [get_imdb_id(mid) for mid in tqdm(df["id"])]

df.to_csv("tmdb_movies.csv", index=False)

100%|██████████| 1958/1958 [04:26<00:00,  7.36it/s]


### **7. Saving both the raw and the processed version**

In [26]:
df.to_csv("data/processed/tmdb_movies.csv", index=False)

In [27]:
tmdb_movies = pd.read_csv("data/processed/tmdb_movies.csv")
tmdb_movies.head()

,id,title,release_date,genres,budget,revenue,vote_average,vote_count,imdb_id
0,2,Ariel,1988-10-21,"['Drama', 'Romance', 'Crime']",0,0,7.115,384,tt0094675
1,3,Shadows in Paradise,1986-10-17,"['Drama', 'Romance']",0,0,7.264,449,tt0092149
2,5,Four Rooms,1995-12-09,['Comedy'],4000000,4257354,5.919,2909,tt0113101
3,6,Judgment Night,1993-10-15,"['Action', 'Crime', 'Drama', 'Thriller']",21000000,12136938,6.446,389,tt0107286
4,8,Life in Loops (A Megacities RMX),2006-01-01,['Documentary'],42000,0,7.300,34,tt0825671
